[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hankpark0706/OL7014/blob/main/notebooks/week6_lap.ipynb)

In [ ]:
%pip install -q gurobipy

In [ ]:
import gurobipy as gp
from gurobipy import GRB

params = {
    "WLSACCESSID": "paste your ACCESSID here",
    "WLSSECRET":   "paste your SECRET here",
    "LICENSEID":   123456,   # the number itself, no quotes
}
env = gp.Env(params=params)

# LAP --- the note's toy

![the network](https://raw.githubusercontent.com/hankpark0706/OL7014/main/notebooks/img/week6_lap_network.png)

$$
\begin{aligned}
\min \quad & 10 x_1 + 20 x_2 + r_1 + r_2 + 2 y_{13} + y_{23} + 3 y_{34} + 4 y_{14} + 100\,(q_3 + q_4) + s_1 + s_2 + s_3 + s_4 \\
\text{s.t.} \quad & y_{13} + y_{14} = r_1 - s_1 \\
& y_{23} = r_2 - s_2 \\
& y_{34} - y_{13} - y_{23} = -5 + q_3 - s_3 \\
& -y_{34} - y_{14} = -8 + q_4 - s_4 \\
& r_1 + r_2 \le 15, \qquad r_1 \le 15 x_1, \qquad r_2 \le 15 x_2, \qquad x_1 + x_2 \le 1 \\
& y, r, s \ge 0, \quad 0 \le q_3 \le 5, \quad 0 \le q_4 \le 8, \quad x_1, x_2 \in \{0,1\}
\end{aligned}
$$

**Math to code**

| math | code |
| :--- | :--- |
| $x_i \in \{0,1\}$ | `addVar(vtype=GRB.BINARY)` |
| $y, r, s \ge 0$ | `addVar()` --- continuous, lower bound `0`, is the default |
| $0 \le q_3 \le 5$ | `addVar(ub=5)` |
| each row | one `addConstr` --- keep a handle (`node4 = ...`), we edit it later |

## Live coding — fill in the blanks

Skeleton for building the model live in class, one comment at a time.

In [ ]:
# Initialize the model gp.Model()


# switches: x1, x2 binary --- lap.addVar(vtype=GRB.BINARY)


# reserves r1, r2 and flows y13, y14, y23, y34 --- continuous, addVar() is enough


# shortfalls q3 (ub=5), q4 (ub=8) and surpluses s1, ..., s4


# objective: open + hold + ship + fail + waste --- lap.setObjective(..., GRB.MINIMIZE)


# conservation, one row per node --- keep handles: node1 = lap.addConstr(...)


# budget, linking, at most p facilities


# Model is set up. Let's optimize --- lap.optimize()


# Let's print the plan: which site opens, the reserve, the four flows, the total cost

## Reference: the completed model

By hand — no dictionaries, no loops, no `quicksum` — so every term matches the math above.

In [ ]:
lap = gp.Model("lap")

# switches -- one per candidate site
x1 = lap.addVar(vtype=GRB.BINARY, name="x1")
x2 = lap.addVar(vtype=GRB.BINARY, name="x2")

# reserves and flows -- tons are divisible, so continuous (lower bound 0 by default)
r1 = lap.addVar(name="r1")
r2 = lap.addVar(name="r2")
y13 = lap.addVar(name="y13")
y14 = lap.addVar(name="y14")
y23 = lap.addVar(name="y23")
y34 = lap.addVar(name="y34")

# shortfalls (capped by demand) and surpluses
q3 = lap.addVar(ub=5, name="q3")
q4 = lap.addVar(ub=8, name="q4")
s1 = lap.addVar(name="s1")
s2 = lap.addVar(name="s2")
s3 = lap.addVar(name="s3")
s4 = lap.addVar(name="s4")

# objective -- open + hold + ship + fail + waste
lap.setObjective(
    10 * x1 + 20 * x2 + r1 + r2
    + 2 * y13 + 4 * y14 + y23 + 3 * y34
    + 100 * (q3 + q4) + s1 + s2 + s3 + s4,
    GRB.MINIMIZE)

# conservation -- MCNF's row at every node (q1 = q2 = 0 since d1 = d2 = 0)
node1 = lap.addConstr(y13 + y14 == r1 - s1, name="node1")
node2 = lap.addConstr(y23 == r2 - s2, name="node2")
node3 = lap.addConstr(y34 - y13 - y23 == -5 + q3 - s3, name="node3")
node4 = lap.addConstr(-y34 - y14 == -8 + q4 - s4, name="node4")

# budget, linking (M = B), at most p = 1 facility
lap.addConstr(r1 + r2 <= 15, name="budget")
lap.addConstr(r1 <= 15 * x1, name="link1")
lap.addConstr(r2 <= 15 * x2, name="link2")
lap.addConstr(x1 + x2 <= 1, name="cardinality")

lap.optimize()

print(f"\nopen: site 1 = {round(x1.X)}, site 2 = {round(x2.X)}   reserves: r1 = {r1.X:g}, r2 = {r2.X:g}")
print(f"flows: y13 = {y13.X:g}, y14 = {y14.X:g}, y23 = {y23.X:g}, y34 = {y34.X:g}")
print(f"shortfall: q3 = {q3.X:g}, q4 = {q4.X:g}   total cost = {lap.ObjVal:g}")

## Round 2 — declare everything integer

Crates ship whole: flip $y, r, q, s$ from continuous to integer and re-optimize. Prediction before running: harder, easier, or no change?

In [ ]:
# flip the vtypes -- no other change
for v in [r1, r2, y13, y14, y23, y34, q3, q4, s1, s2, s3, s4]:
    v.VType = GRB.INTEGER

lap.optimize()

print(f"\nopen: site 1 = {round(x1.X)}, site 2 = {round(x2.X)}   reserves: r1 = {r1.X:g}, r2 = {r2.X:g}")
print(f"flows: y13 = {y13.X:g}, y14 = {y14.X:g}, y23 = {y23.X:g}, y34 = {y34.X:g}")
print(f"total cost = {lap.ObjVal:g}   -- same plan, same 65")

**Same plan, same 65 — and the continuous solve was already integer.** Why: fix $x$, and the linking rows become plain bounds. What is left in $(r, y, q, s)$ is an MCNF on an augmented graph — $r_i$ is an arc from a super-source (the budget row is its capacity $B$), $q_i$ and $s_i$ are shortfall and disposal arcs. Every column holds one $+1$ and one $-1$: **TU** by week 5's checklist ($M_1$ = all rows, $M_2 = \emptyset$). Integer data $\Rightarrow$ integer corners. Integrality on the flow side never binds; branch \& bound only ever has to work on $x$.

## Round 3 — the contrast: relax $x$ instead

The hardness is not symmetric. Relax **everything** (in particular the switches) and watch what breaks.

In [ ]:
rel = lap.relax()   # drops every integrality declaration
rel.optimize()

print(f"\nx1 = {rel.getVarByName('x1').X:.4f}   x2 = {rel.getVarByName('x2').X:.4f}")
print(f"r1 = {rel.getVarByName('r1').X:g}   r2 = {rel.getVarByName('r2').X:g}")
print(f"z_LP = {rel.ObjVal:g}   <   z_IP = 65   -- half-open facilities: the gap lives in x, not in y")

## The caveat — integer data

The TU guarantee needs an integer right-hand side. Make node 4's demand fractional, $d_4 = 8.5$, and integer flows start to cost real money. ($q, s$ stay continuous — shortfall in tons can be fractional; make them integer too and the model goes **infeasible**: try it.)

In [ ]:
# d4 = 8.5 -- edit the row and the shortfall cap
node4.RHS = -8.5
q4.UB = 8.5

# continuous flows first
for v in [r1, r2, y13, y14, y23, y34, q3, q4, s1, s2, s3, s4]:
    v.VType = GRB.CONTINUOUS
lap.optimize()
z_cont = lap.ObjVal
print(f"\ncontinuous flows: y14 = {y14.X:g}, r1 = {r1.X:g}, cost = {z_cont:g}")

# now whole crates: y and r integer
for v in [r1, r2, y13, y14, y23, y34]:
    v.VType = GRB.INTEGER
lap.optimize()
print(f"integer flows:    y14 = {y14.X:g}, s4 = {s4.X:g}, cost = {lap.ObjVal:g}")
print(f"\nintegrality now binds: {lap.ObjVal:g} > {z_cont:g} -- ship the 9th crate and waste half of it")

**Takeaway.** With integer data, declaring $y, q, s$ integer or continuous does not affect the LAP — the flow side is TU once $x$ is fixed, so its corners are integer for free. Declare integrality only where it binds: that is why the letters split in week 5 — $x$ integer, $y$ continuous.